# SISO control analysis: NumPy / SciPy, python-control, minilink

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/siso_transfer_function_analysis.ipynb)

The basic commands to analyze a single-input single-output control loop, on a mass–spring–damper:

1. **Plant**: the transfer function $H(s) = y/u$.
2. **Open loop**: step response, poles and zeros, frequency response.
3. **Closed loop**: a controller $C(s) = u/e$, the loop gain $L(s) = C(s)H(s)$ (root locus, Bode plot), then $CL(s) = y/r$ (step response, poles and zeros).

The same sequence three times: with **NumPy / SciPy / Matplotlib** only, with **python-control**, then with **minilink**. When a library has no tool for a step, that step is simply skipped.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## Part 1 — NumPy, SciPy and Matplotlib

No $s$ variable: a transfer function is a list of numerator coefficients and a list of denominator coefficients, in decreasing powers of $s$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

### Plant

$H(s) = \dfrac{y}{u} = \dfrac{1}{m s^2 + b s + k}$

In [ ]:
m, b, k = 10.0, 2.0, 10.0

num_H, den_H = [1.0], [m, b, k]
H = signal.TransferFunction(num_H, den_H)
H  # SciPy normalizes the denominator

### Open-loop analysis

Time response to a step $u = 1$ (the transient), location of the poles and zeros, then frequency response to $u = \sin(\omega t)$ (the steady state).

In [ ]:
t, y = signal.step(H)

plt.plot(t, y)
plt.xlabel("t [s]")
plt.ylabel("y")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(H.poles.real, H.poles.imag, "x", label="poles")
plt.plot(H.zeros.real, H.zeros.imag, "o", label="zeros")
plt.axis("equal")
plt.xlabel("Re")
plt.ylabel("Im")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
w, mag, phase = signal.bode(H)

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].semilogx(w, mag)
ax[1].semilogx(w, phase)
ax[0].set_ylabel("|H| [dB]")
ax[1].set_ylabel("phase [deg]")
ax[1].set_xlabel("ω [rad/s]")
ax[0].grid(True, which="both")
ax[1].grid(True, which="both")
plt.show()

### Closed-loop analysis

PID controller, $C(s) = \dfrac{u}{e} = k_p + k_d s + \dfrac{k_i}{s} = \dfrac{k_d s^2 + k_p s + k_i}{s}$, and loop gain $L(s) = \dfrac{y}{e} = C(s)H(s)$.

SciPy has no transfer-function algebra: the product $C H$ is computed on the polynomials with `np.polymul`.

In [ ]:
kp, kd, ki = 1.0, 1.0, 1.0

num_C, den_C = [kd, kp, ki], [1.0, 0.0]
num_L, den_L = np.polymul(num_C, num_H), np.polymul(den_C, den_H)
L = signal.TransferFunction(num_L, den_L)
L

Root locus: no tool in SciPy, step skipped.

In [ ]:
w, mag, phase = signal.bode(L)

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].semilogx(w, mag)
ax[1].semilogx(w, phase)
ax[0].set_ylabel("|L| [dB]")
ax[1].set_ylabel("phase [deg]")
ax[1].set_xlabel("ω [rad/s]")
ax[0].grid(True, which="both")
ax[1].grid(True, which="both")
plt.show()

Closed-loop transfer function, $CL(s) = \dfrac{y}{r} = \dfrac{L}{1 + L} = \dfrac{N_L}{D_L + N_L}$: a sum of polynomials with `np.polyadd`, already in minimal form.

In [ ]:
CL = signal.TransferFunction(num_L, np.polyadd(den_L, num_L))
CL

In [ ]:
t, y = signal.step(CL)

plt.plot(t, y)
plt.xlabel("t [s]")
plt.ylabel("y")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(CL.poles.real, CL.poles.imag, "x", label="poles")
plt.plot(CL.zeros.real, CL.zeros.imag, "o", label="zeros")
plt.axis("equal")
plt.xlabel("Re")
plt.ylabel("Im")
plt.legend()
plt.grid(True)
plt.show()

## Part 2 — python-control

The [python-control](https://python-control.readthedocs.io/en/0.10.2/) library mirrors the MATLAB commands: an $s$ variable, transfer-function algebra and one-line plots.

In [ ]:
import importlib.util

if importlib.util.find_spec("control") is None:
    get_ipython().system("pip install -q control")

In [ ]:
import control as ct

### Plant

In [ ]:
s = ct.tf("s")

m, b, k = 10.0, 2.0, 10.0

H = 1 / (m * s**2 + b * s + k)
H

### Open-loop analysis

In [ ]:
ct.step_response(H).plot();

In [ ]:
ct.pzmap(H);

In [ ]:
ct.bode_plot(H);

### Closed-loop analysis

In [ ]:
kp, kd, ki = 1.0, 1.0, 1.0

C = kp + kd * s + ki / s
L = C * H
L

In [ ]:
ct.root_locus(L);

In [ ]:
ct.bode_plot(L);

`minreal` cancels the common factors that computing $L / (1 + L)$ duplicates.

In [ ]:
CL = ct.minreal(L / (1 + L), verbose=False)
CL

In [ ]:
ct.step_response(CL).plot();

In [ ]:
ct.pzmap(CL);

## Part 3 — minilink

In [minilink](https://github.com/alx87grd/minilink), a transfer function is a block: `>>` puts two blocks in series, `@` closes the loop with $e = r - y$, and every analysis tool is a `plot_…` verb of the block. No $s$ variable (coefficient lists, as in Part 1) and no `minreal`: `@` builds the closed loop as a diagram, so there is nothing to cancel.

In [ ]:
from minilink import PID, TransferFunction

### Plant

In [ ]:
m, b, k = 10.0, 2.0, 10.0

H = TransferFunction([1.0], [m, b, k])

### Open-loop analysis

In [ ]:
H.plot_step_response()

In [ ]:
H.plot_pzmap()

In [ ]:
H.plot_bode(margins=False)

### Closed-loop analysis

minilink's `PID` filters its derivative, $k_d \dfrac{s}{\tau s + 1}$: one extra fast pole at $-1/\tau = -100$ ($\tau = 0.01$ s), visible far to the left on the root locus and on the pole-zero map. We zoom in near the origin.

In [ ]:
kp, kd, ki = 1.0, 1.0, 1.0

C = PID(Kp=kp, Ki=ki, Kd=kd, tau=0.01)
L = C >> H

In [ ]:
locus = L.plot_root_locus(show=False)
locus.axes.set_xlim(-1.5, 0.5)
locus.axes.set_ylim(-1.5, 1.5);

In [ ]:
L.plot_bode()

In [ ]:
CL = C @ H
CL.plot_diagram()

In [ ]:
CL.plot_step_response()

In [ ]:
pzmap = CL.plot_pzmap(show=False)
pzmap.axes.set_xlim(-1.5, 0.5)
pzmap.axes.set_ylim(-1.5, 1.5);